# Exercises XP: Vector Databases and RAG

## What you'll learn
- Vector search strategies (KNN, ANN) and evaluation.
- Vector database utility (similarity search, RAG).
- Differences between vector DBs, libraries, and plugins.
- Best practices for vector store usage and performance.
- How LMs use context; embedding generation and storage.
- Querying vector stores and applying LMs for QA with retrieved context.

## What you'll build
A functional RAG pipeline with FAISS and ChromaDB, plus QA over retrieved context using a Hugging Face model.

## 0. Setup
The course instructions pin old versions (`faiss-cpu==1.7.4`, `chromadb==0.3.21`, `numpy<2`, `pydantic<2`). Those wheels do not exist for recent Python versions (Colab now runs Python ≥ 3.11 with NumPy 2), so we install current releases instead. `faiss-cpu ≥ 1.8` ships its own OpenMP runtime, so `apt install libomp-dev` is no longer required, and recent FAISS/Chroma work with NumPy 2. We keep `transformers < 5` and `sentence-transformers < 6`, which are compatible with each other.

In [ ]:
%pip install -q -U faiss-cpu chromadb "sentence-transformers<6" "transformers<5" accelerate

In [ ]:
import os
os.environ["USE_TF"] = "0"                    # transformers: use PyTorch only
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["ANONYMIZED_TELEMETRY"] = "False"

import json, time
from pathlib import Path
import numpy as np
import pandas as pd
import faiss
import torch
from sentence_transformers import SentenceTransformer, InputExample
import chromadb
from chromadb.config import Settings
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM, pipeline
from IPython.display import display
import chromadb as _c, sentence_transformers as _s, transformers as _t

os.makedirs("cache", exist_ok=True)
pd.set_option("display.max_colwidth", 110)
print("faiss", faiss.__version__, "| chromadb", _c.__version__, "| sentence-transformers", _s.__version__,
      "| transformers", _t.__version__, "| numpy", np.__version__)

## 🌟 Exercise 1 · Data loading and preparation

In [ ]:
# The CSV is ';'-separated and zipped; pandas reads the zip directly (local copy if present, otherwise the course repo)
LOCAL = Path("labelled_newscatcher_dataset.csv")
URL = ("https://media.githubusercontent.com/media/devtlv/Datasets-GEN-AI-Bootcamp/main/Week%205/"
       "Day%204%20-%20Diving%20Deep%20into%20Vector%20Databases%20and%20RAG%20Chatbots/labelled_newscatcher_dataset.zip")
data_path = LOCAL if LOCAL.exists() else URL
pdf = pd.read_csv(data_path, sep=";")

# Add a unique identifier column
if "id" not in pdf.columns:
    pdf["id"] = range(len(pdf))

display(pdf)

In [ ]:
print("Shape:", pdf.shape)
print("\nColumn types:"); print(pdf.dtypes)
print("\nMissing values per column:"); print(pdf.isna().sum())
print("\nArticles per topic:"); print(pdf["topic"].value_counts())
print("\nLanguages:", pdf["lang"].unique(), "| domains:", pdf["domain"].nunique())
print("Title length (words):", pdf["title"].str.split().str.len().describe().round(1).to_dict())

**Observations.** 108 774 English news articles with 6 columns plus our `id`:
- `topic`: the label (8 classes, 15 000 articles each except SCIENCE with 3 774);
- `link`, `domain`: source;
- `published_date`: a string, not parsed;
- `title`: the text we will embed;
- `lang`: always `en`.

There are **no missing values**. Titles are short (about 11 words on average), which suits sentence-embedding models. The file is sorted by topic, so the first 1 000 rows contain only SCIENCE, TECHNOLOGY and HEALTH articles; we keep this subset as asked, which is fine for queries such as *animal* or *space*.

In [ ]:
# Create a manageable subset (first 1000 rows)
pdf_subset = pdf.head(1000).copy()
print(pdf_subset["topic"].value_counts())
pdf_subset[["id", "title"]].head()

## 🌟 Exercise 2 · Vectorization with Sentence Transformers

In [ ]:
def example_create_fn(doc1: pd.Series) -> InputExample:
    """
    Helper function that outputs a sentence_transformer guid, label, and text.
    """
    return InputExample(guid=str(doc1.name), texts=[doc1["title"]], label=0.0)

# one InputExample per news title (guid = row index, label unused for inference)
faiss_train_examples = pdf_subset.apply(lambda x: example_create_fn(x), axis=1).tolist()
faiss_train_examples[:10]

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")
print(model)

titles_list = pdf_subset["title"].tolist()
t0 = time.time()
faiss_title_embedding = model.encode(titles_list, convert_to_numpy=True, show_progress_bar=False, batch_size=64)
print(f"Encoded {len(titles_list)} titles in {time.time()-t0:.1f}s")
len(faiss_title_embedding), len(faiss_title_embedding[0])

We get **1 000 embeddings** (one per title), each of **384 dimensions**: `all-MiniLM-L6-v2` maps any sentence to a dense 384-d vector, where semantically close titles get close vectors.

## 🌟 Exercise 3 · FAISS indexing and search

In [ ]:
pdf_to_index = pdf_subset
id_index = pdf_to_index["id"].to_numpy().astype(np.int64)

content_encoded_normalized = faiss_title_embedding.astype("float32").copy()
faiss.normalize_L2(content_encoded_normalized)          # unit-length vectors -> inner product = cosine similarity
print("norms after normalisation:", np.linalg.norm(content_encoded_normalized, axis=1)[:5].round(4))

index_content = faiss.IndexIDMap(faiss.IndexFlatIP(len(faiss_title_embedding[0])))
index_content.add_with_ids(content_encoded_normalized, id_index)
index_content.ntotal

In [ ]:
def search_content(query, pdf_to_index, k=3):
    query_vector = model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(query_vector)

    # Perform the search
    top_k = index_content.search(query_vector, k)
    ids = top_k[1][0].tolist()
    similarities = top_k[0][0].tolist()

    # keep FAISS' ranking order (isin() alone would return the rows in DataFrame order)
    results = pdf_to_index.set_index("id").loc[ids].reset_index()
    results["similarities"] = similarities
    return results

display(search_content("animal", pdf_to_index, k=5)[["id", "topic", "title", "similarities"]])

In [ ]:
for q in ["space exploration", "covid vaccine trial", "new smartphone release", "climate change"]:
    print(f"Query: {q!r}")
    display(search_content(q, pdf_to_index, k=3)[["topic", "title", "similarities"]])

### Going further: exact KNN vs approximate (ANN) search on the full dataset
`IndexFlatIP` is an **exact** k-nearest-neighbour search: it compares the query with every vector, so its cost is O(N·d). On larger collections we use **approximate** indexes.
- **IVF** (inverted file): k-means clusters; only `nprobe` clusters are scanned.
- **HNSW**: a navigable small-world graph.

Below we embed all 108 774 titles and compare the indexes on **recall@10**, i.e. the share of the exact top-10 that the ANN index also returns, and on query latency.

In [ ]:
t0 = time.time()
all_emb = model.encode(pdf["title"].tolist(), convert_to_numpy=True, batch_size=256, show_progress_bar=False).astype("float32")
faiss.normalize_L2(all_emb)
print(f"Embedded {len(all_emb)} titles in {time.time()-t0:.0f}s -> {all_emb.shape}, {all_emb.nbytes/1e6:.0f} MB")

d = all_emb.shape[1]
rng = np.random.default_rng(0)
query_ids = rng.choice(len(all_emb), 500, replace=False)
queries = all_emb[query_ids]

def bench(index, name, build_time):
    t0 = time.time(); _, I = index.search(queries, 11); ms = (time.time() - t0) / len(queries) * 1000
    return {"index": name, "build (s)": round(build_time, 1), "query (ms)": round(ms, 3), "I": I}

rows = []
t0 = time.time(); flat = faiss.IndexFlatIP(d); flat.add(all_emb); rows.append(bench(flat, "Flat (exact KNN)", time.time() - t0))
truth = rows[0]["I"]

t0 = time.time()
ivf = faiss.IndexIVFFlat(faiss.IndexFlatIP(d), d, 256, faiss.METRIC_INNER_PRODUCT)
ivf.train(all_emb); ivf.add(all_emb); bt = time.time() - t0
for nprobe in [1, 4, 16]:
    ivf.nprobe = nprobe; rows.append(bench(ivf, f"IVF256, nprobe={nprobe}", bt))

t0 = time.time()
hnsw = faiss.IndexHNSWFlat(d, 32, faiss.METRIC_INNER_PRODUCT); hnsw.add(all_emb); bt = time.time() - t0
for ef in [16, 64]:
    hnsw.hnsw.efSearch = ef; rows.append(bench(hnsw, f"HNSW32, efSearch={ef}", bt))

for r in rows:
    r["recall@10"] = np.mean([len(set(a[1:]) & set(b[1:])) / 10 for a, b in zip(r["I"], truth)])  # [1:] skips the query itself
ann = pd.DataFrame(rows).drop(columns="I")
ann

## 🌟 Exercise 4 · ChromaDB collection and querying

In [ ]:
chroma_client = chromadb.Client(Settings(anonymized_telemetry=False))
collection_name = "my_news"

# If a collection with the same name exists, delete it to avoid conflicts
if any(c.name == collection_name for c in chroma_client.list_collections()):
    chroma_client.delete_collection(name=collection_name)

print(f"Creating collection: '{collection_name}'")
collection = chroma_client.create_collection(name=collection_name)

display(pdf_subset)

collection.add(
    documents=pdf_subset["title"][:100].tolist(),
    metadatas=[{"topic": topic} for topic in pdf_subset["topic"][:100].tolist()],
    ids=[f"id{x}" for x in pdf_subset["id"][:100].tolist()],
)
print("Documents in collection:", collection.count())

In [ ]:
results = collection.query(query_texts=["space"], n_results=10)
print(json.dumps(results, indent=4))

In [ ]:
# Same results as a table (Chroma's default space is squared L2: smaller distance = more similar)
pd.DataFrame({"id": results["ids"][0], "distance": results["distances"][0],
              "topic": [m["topic"] for m in results["metadatas"][0]], "title": results["documents"][0]})

Chroma embedded the 100 titles **automatically**, with its default embedding function (`all-MiniLM-L6-v2` run through ONNX). For *space* it ranks first the titles about NASA, astronauts, Mars, planets and the Moon. The returned JSON contains, for each query, the `ids`, `documents`, `metadatas` and `distances` of the 10 nearest neighbours.

Chroma also does what a plain library like FAISS does not do: it stores documents and **metadata**, and it can **filter** on that metadata. Below we restrict the search to a topic, and we create a second collection that uses cosine distance and reuses the embeddings we already computed.

In [ ]:
print("Topics among the 100 docs:", pd.Series([m["topic"] for m in collection.get()["metadatas"]]).value_counts().to_dict())
filtered = collection.query(query_texts=["new discovery"], n_results=3, where={"topic": "TECHNOLOGY"})
print("\n'new discovery' restricted to TECHNOLOGY:")
for doc, dist in zip(filtered["documents"][0], filtered["distances"][0]):
    print(f"  {dist:.3f}  {doc}")

# bring-your-own embeddings + cosine distance on the full 1000-title subset
if any(c.name == "my_news_cosine" for c in chroma_client.list_collections()):
    chroma_client.delete_collection("my_news_cosine")
col_cos = chroma_client.create_collection("my_news_cosine", metadata={"hnsw:space": "cosine"})
col_cos.add(ids=[str(i) for i in pdf_subset["id"]], embeddings=faiss_title_embedding.tolist(),
            documents=pdf_subset["title"].tolist(), metadatas=[{"topic": t} for t in pdf_subset["topic"]])
res_cos = col_cos.query(query_embeddings=model.encode(["space"]).tolist(), n_results=5)
print("\n'space' on 1000 titles (cosine distance = 1 - cosine similarity):")
for doc, dist in zip(res_cos["documents"][0], res_cos["distances"][0]):
    print(f"  {dist:.3f}  {doc}")

## 🌟 Exercise 5 · Question answering with a Hugging Face model

The exercise asks for a **causal LM** (`AutoModelForCausalLM`, e.g. GPT-2), while the template suggests `google/flan-t5-small`. We use both:
- **`Qwen/Qwen2.5-0.5B-Instruct`**, a small (0.5 B parameters) instruction-tuned causal LM. It runs on CPU, and unlike GPT-2 it was trained to follow instructions, so it can actually *answer* a question from a context. GPT-2 only continues text.
- **`google/flan-t5-small`**, the template's seq2seq alternative, for comparison.

In [ ]:
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
lm_model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float32)

pipe = pipeline(
    "text-generation",
    model=lm_model,
    tokenizer=tokenizer,
    max_new_tokens=512,   # maximum number of tokens to generate
    device_map="auto",    # GPU if available, else CPU
)
print(f"{model_id}: {sum(p.numel() for p in lm_model.parameters())/1e6:.0f}M parameters on {lm_model.device}")

In [ ]:
question = "What's the latest news on space development?"
context = " ".join([f"#{str(i)}" for i in results["documents"][0]])     # exercise template (as given)
print("Context as built by the template:", context[:200], "...")

Note: the template's `f"#{str(i)}"` **replaces** each document with `#` followed by the document itself. It is easier for the model if each document is numbered and placed on its own line, so we build the context with `f"#{n} {doc}"` instead.

In [ ]:
context = "\n".join(f"#{n} {doc}" for n, doc in enumerate(results["documents"][0], 1))
prompt_template = f"Relevant context: {context}\n\n The user's question: {question}"
print(prompt_template)

In [ ]:
def generate(prompt, max_new_tokens=200):
    # chat template = the format the instruct model was trained on; greedy decoding for reproducibility
    messages = [{"role": "system", "content": "You are a helpful assistant. Answer using only the provided news headlines. "
                                              "If the context does not contain the answer, say so."},
                {"role": "user", "content": prompt}]
    out = pipe(messages, max_new_tokens=max_new_tokens, do_sample=False)
    return out[0]["generated_text"][-1]["content"]

t0 = time.time()
lm_response = pipe([{"role": "user", "content": prompt_template}], do_sample=False)
print(lm_response[0]["generated_text"][-1]["content"])
print(f"\n[{time.time()-t0:.1f}s]")

### 6. Experiment with different prompts and context windows

In [ ]:
def rag_answer(question, k=3, collection=col_cos, show_context=True):
    hits = collection.query(query_embeddings=model.encode([question]).tolist(), n_results=k)
    ctx = "\n".join(f"#{n} {doc}" for n, doc in enumerate(hits["documents"][0], 1))
    if show_context:
        print(f"--- context (k={k}) ---\n{ctx}")
    prompt = f"Relevant context:\n{ctx}\n\nThe user's question: {question}\nAnswer in 2-3 sentences, citing the headline numbers."
    return generate(prompt)

q = "What's the latest news on space development?"
for k in [1, 3, 10]:
    t0 = time.time()
    print(f"\n===== k = {k} =====")
    print("ANSWER:", rag_answer(q, k=k), f"\n[{time.time()-t0:.1f}s]")

In [ ]:
# Without retrieval: the model can only use what it memorised during pre-training
print("NO CONTEXT:", generate(q + " Answer in 2-3 sentences.", max_new_tokens=150))

In [ ]:
for q in ["What did researchers find about COVID-19?",
          "Are there any news about animals?",
          "Who won the football match yesterday?"]:     # not covered by the SCIENCE/TECH/HEALTH subset
    print(f"\n===== {q} =====")
    print("ANSWER:", rag_answer(q, k=5))

In [ ]:
# The template's alternative: a small seq2seq model (flan-t5-small, 77M parameters)
t5_pipe = pipeline("text2text-generation", model="google/flan-t5-small", max_new_tokens=64)
context_docs = results["documents"][0][:3]
context = " ".join(context_docs)
prompt = f"Answer the question using only the context.\nContext: {context}\nQuestion: {question}\nAnswer:\n"
response = t5_pipe(prompt)[0]["generated_text"]
print(response)

ANALYSIS_RAG